In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.naive_bayes import GaussianNB, MultinomialNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [3]:
# STEP 1 : LOAD DATASET

data = pd.read_csv("adult.csv", header=None)
data.columns = [
    "Age",
    "Workclass",
    "fnlgt",
    "Education",
    "Education_num",
    "Marital_Status",
    "Occupation",
    "Relationship",
    "Race",
    "Sex",
    "Capital_Gain",
    "Capital_Loss",
    "Hours_per_week",
    "Native_Country",
    "Income"
]
print("First Five Records")
print(data.head())
print("\nDataset Shape :", data.shape)

First Five Records
   Age          Workclass   fnlgt   Education  Education_num  \
0   39          State-gov   77516   Bachelors             13   
1   50   Self-emp-not-inc   83311   Bachelors             13   
2   38            Private  215646     HS-grad              9   
3   53            Private  234721        11th              7   
4   28            Private  338409   Bachelors             13   

        Marital_Status          Occupation    Relationship    Race      Sex  \
0        Never-married        Adm-clerical   Not-in-family   White     Male   
1   Married-civ-spouse     Exec-managerial         Husband   White     Male   
2             Divorced   Handlers-cleaners   Not-in-family   White     Male   
3   Married-civ-spouse   Handlers-cleaners         Husband   Black     Male   
4   Married-civ-spouse      Prof-specialty            Wife   Black   Female   

   Capital_Gain  Capital_Loss  Hours_per_week  Native_Country  Income  
0          2174             0              40   U

In [4]:
# STEP 2 : REMOVE MISSING VALUES

data.replace(" ?", np.nan, inplace=True)
data.dropna(inplace=True)
print("\nDataset Shape After Removing Missing Values")
print(data.shape)


Dataset Shape After Removing Missing Values
(30162, 15)


In [5]:
# STEP 3 : ENCODE TARGET VARIABLE
label = LabelEncoder()
data["Income"] = label.fit_transform(data["Income"])

In [6]:
# STEP 4 : GROUP OTHER COUNTRIES
data["Native_Country"] = data["Native_Country"].apply(
    lambda x: x if x == " United-States" else "Other")

In [7]:
# STEP 5 : ONE HOT ENCODING
categorical_columns = data.select_dtypes(include="object").columns
data = pd.get_dummies(data, columns=categorical_columns)
print("\nDataset After Encoding")
print(data.head())


Dataset After Encoding
   Age   fnlgt  Education_num  Capital_Gain  Capital_Loss  Hours_per_week  \
0   39   77516             13          2174             0              40   
1   50   83311             13             0             0              13   
2   38  215646              9             0             0              40   
3   53  234721              7             0             0              40   
4   28  338409             13             0             0              40   

   Income  Workclass_ Federal-gov  Workclass_ Local-gov  Workclass_ Private  \
0       0                   False                 False               False   
1       0                   False                 False               False   
2       0                   False                 False                True   
3       0                   False                 False                True   
4       0                   False                 False                True   

   ...  Relationship_ Wife  Race_ Amer

In [8]:
# STEP 6 : SPLIT FEATURES AND LABEL
X = data.drop("Income", axis=1)
y = data["Income"]

# STEP 7 : TRAIN TEST SPLIT
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [9]:
# STEP 8 : GAUSSIAN NAIVE BAYES

print("\n==============================")
print("GAUSSIAN NAIVE BAYES")
print("==============================")
gnb = GaussianNB()
gnb.fit(X_train, y_train)
gnb_prediction = gnb.predict(X_test)

print("\nAccuracy :",
      round(accuracy_score(y_test, gnb_prediction)*100,2),"%")

print("\nConfusion Matrix")

print(confusion_matrix(y_test, gnb_prediction))

print("\nClassification Report")
print(classification_report(y_test, gnb_prediction))


GAUSSIAN NAIVE BAYES

Accuracy : 79.11 %

Confusion Matrix
[[4282  221]
 [1039  491]]

Classification Report
              precision    recall  f1-score   support

           0       0.80      0.95      0.87      4503
           1       0.69      0.32      0.44      1530

    accuracy                           0.79      6033
   macro avg       0.75      0.64      0.65      6033
weighted avg       0.78      0.79      0.76      6033



In [10]:
# STEP 9 : MULTINOMIAL NAIVE BAYES
print("\n==============================")
print("MULTINOMIAL NAIVE BAYES")
print("==============================")
mnb = MultinomialNB()
mnb.fit(X_train, y_train)
mnb_prediction = mnb.predict(X_test)
print("\nAccuracy :",
      round(accuracy_score(y_test, mnb_prediction)*100,2),"%")
print("\nConfusion Matrix")
print(confusion_matrix(y_test, mnb_prediction))
print("\nClassification Report")
print(classification_report(y_test, mnb_prediction))


MULTINOMIAL NAIVE BAYES

Accuracy : 77.46 %

Confusion Matrix
[[4307  196]
 [1164  366]]

Classification Report
              precision    recall  f1-score   support

           0       0.79      0.96      0.86      4503
           1       0.65      0.24      0.35      1530

    accuracy                           0.77      6033
   macro avg       0.72      0.60      0.61      6033
weighted avg       0.75      0.77      0.73      6033



In [11]:
# STEP 10 : ACCURACY COMPARISON
comparison = pd.DataFrame({
    "Model":[
        "Gaussian Naive Bayes",
        "Multinomial Naive Bayes"
    ],
    "Accuracy (%)":[
        round(accuracy_score(y_test,gnb_prediction)*100,2),
        round(accuracy_score(y_test,mnb_prediction)*100,2)
    ]
})
print("\n==============================")
print("MODEL COMPARISON")
print("==============================")
print(comparison)


MODEL COMPARISON
                     Model  Accuracy (%)
0     Gaussian Naive Bayes         79.11
1  Multinomial Naive Bayes         77.46


In [12]:
# STEP 11 : PREDICT NEW SAMPLE
sample = X_test.iloc[[0]]
print("\nActual Class :", y_test.iloc[0])
print("Gaussian Prediction :", gnb.predict(sample)[0])
print("Multinomial Prediction :", mnb.predict(sample)[0])
print("\nProbability (Gaussian)")
print(gnb.predict_proba(sample))
print("\nProbability (Multinomial)")
print(mnb.predict_proba(sample))


Actual Class : 1
Gaussian Prediction : 0
Multinomial Prediction : 0

Probability (Gaussian)
[[0.98423812 0.01576188]]

Probability (Multinomial)
[[1. 0.]]
